In [1]:
# ============================================
# CYBERSECURITY RAG CHATBOT USING GROQ + GRADIO
# ============================================

# Install required libraries
!pip install groq gradio sentence-transformers faiss-cpu pypdf

# Imports
import os
import gradio as gr
import faiss
import numpy as np
from groq import Groq
from sentence_transformers import SentenceTransformer

# ============================================
# SET YOUR GROQ API KEY
# ============================================

GROQ_API_KEY = "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"   # Replace with your real API key
client = Groq(api_key=GROQ_API_KEY)

# ============================================
# LOAD EMBEDDING MODEL
# ============================================

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# ============================================
# SAMPLE CYBERSECURITY KNOWLEDGE BASE
# You can replace these with your own notes/files
# ============================================

documents = [
    """
    Risk assessment is the process of identifying vulnerabilities,
    threats, and weaknesses in a system.
    """,

    """
    Risk management is the process of reducing, controlling,
    and handling identified risks after assessment.
    """,

    """
    Firewall is a network security device that monitors
    incoming and outgoing traffic.
    """,

    """
    IDS stands for Intrusion Detection System.
    IPS stands for Intrusion Prevention System.
    """,

    """
    VPN stands for Virtual Private Network.
    It encrypts communication over public networks.
    """,

    """
    CIA Triad includes Confidentiality, Integrity, and Availability.
    """
]

# ============================================
# CREATE VECTOR DATABASE
# ============================================

doc_embeddings = embedding_model.encode(documents)

dimension = doc_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(doc_embeddings).astype("float32"))

# ============================================
# RETRIEVAL FUNCTION
# ============================================

def retrieve_context(query, k=2):
    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"), k
    )

    retrieved_docs = [documents[i] for i in indices[0]]
    return "\n".join(retrieved_docs)

# ============================================
# CHAT FUNCTION
# ============================================

def cybersecurity_chatbot(user_question):
    context = retrieve_context(user_question)

    prompt = f"""
You are a cybersecurity assistant chatbot.

Use the provided cybersecurity context to answer clearly
in easy wording.

Context:
{context}

Question:
{user_question}
"""

    completion = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0.7,
        max_tokens=1024
    )

    answer = completion.choices[0].message.content
    return answer

# ============================================
# GRADIO FRONTEND
# ============================================

interface = gr.Interface(
    fn=cybersecurity_chatbot,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Ask cybersecurity question here..."
    ),
    outputs=gr.Textbox(lines=10),
    title="Cybersecurity RAG Chatbot",
    description="Ask cybersecurity questions using Groq + Llama 3.1 + RAG"
)

interface.launch(share=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 47.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 15.3 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fc6aef3342c545b1c7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
